# Paper #13: HMI Implementation / HMI 구현
Scherrer et al. (2012) — SDO/HMI 기기의 핵심 개념 구현

이 노트북은 SDO/HMI 기기의 핵심 측정 원리와 데이터 처리 개념을 구현합니다.
This notebook implements the core measurement principles and data processing concepts of the SDO/HMI instrument.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import special, signal
from matplotlib.gridspec import GridSpec

# Standard plot settings
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

print("All imports successful.")

## Part 1: MDI vs HMI 비교 / MDI vs HMI Comparison

MDI(1995-2011)에서 HMI(2010-)로의 발전은 태양 자기장 관측의 획기적 도약을 의미합니다.
The evolution from MDI (1995-2011) to HMI (2010-) represents a quantum leap in solar magnetic field observation.

In [ ]:
# =============================================================
# MDI vs HMI Comparison
# =============================================================

# Comparison parameters
categories = [
    "Pixels\n(ratio)",
    "Spatial\nResolution",
    "Magnetic\nSensitivity",
    "Telemetry\nRate",
    "Bandpass\nImprovement",
]
mdi_values = ["1024x1024", '4"', "20 Mx/cm2", "5 Mbps", "94 mA"]
hmi_values = ["4096x4096", '1"', "~10 Mx/cm2", "55 Mbps", "76 mA"]
improvement = [16, 4, 1.75, 11, 1.24]

# Print comparison table
print("=" * 60)
print("MDI vs HMI Comparison Table")
print("=" * 60)
header = "{:<20s} {:<15s} {:<15s} {:<10s}"
print(header.format("Parameter", "MDI", "HMI", "Factor"))
print("-" * 60)
row_fmt = "{:<20s} {:<15s} {:<15s} {:<10.1f}x"
for i in range(len(categories)):
    name = categories[i].replace("\n", " ")
    print(row_fmt.format(name, mdi_values[i], hmi_values[i], improvement[i]))
print("=" * 60)

# Bar chart of improvement factors
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(categories))
colors = ["#2196F3", "#4CAF50", "#FF9800", "#E91E63", "#9C27B0"]
bars = ax.bar(x, improvement, color=colors, width=0.6, edgecolor="black")

# Add value labels on bars
for bar, val in zip(bars, improvement):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.2,
        str(val) + "x",
        ha="center",
        fontweight="bold",
        fontsize=12,
    )

ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=9)
ax.set_ylabel("Improvement Factor")
ax.set_title("HMI Improvements over MDI")
ax.set_ylim(0, 19)
ax.axhline(y=1, color="gray", linestyle="--", alpha=0.5, label="No improvement")
ax.legend()
plt.tight_layout()
plt.show()

## Part 2: Zeeman 분리 / Zeeman Splitting

HMI는 Fe I 6173 A 흡수선의 Zeeman 분리를 측정하여 자기장을 관측합니다.
HMI measures the Zeeman splitting of the Fe I 6173 A absorption line to observe the magnetic field.

$$\Delta\lambda = 4.67 \times 10^{-13} \lambda_0^2 \, g_{\rm eff} \, B$$

In [ ]:
# =============================================================
# Zeeman Splitting: Fe I 6173 vs Ni I 6768
# =============================================================

def zeeman_splitting_nm(lambda_nm, g_eff, B_gauss):
    """Compute Zeeman splitting in nm.

    Args:
        lambda_nm: Wavelength in nm.
        g_eff: Effective Lande g-factor.
        B_gauss: Magnetic field in Gauss.

    Returns:
        Splitting delta_lambda in nm.
    """
    return 4.67e-13 * lambda_nm**2 * g_eff * B_gauss

# Line parameters
fe_lambda = 617.3  # nm
fe_g = 2.5
ni_lambda = 676.8  # nm
ni_g = 1.43

B = np.linspace(10, 5000, 500)

# Compute splitting
fe_split = zeeman_splitting_nm(fe_lambda, fe_g, B)
ni_split = zeeman_splitting_nm(ni_lambda, ni_g, B)

# HMI bandpass = 76 mA = 0.0076 nm
hmi_bandpass = 0.0076

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(B, fe_split * 1000, "b-", linewidth=2, label="Fe I 6173 (g=2.5, HMI)")
ax.plot(B, ni_split * 1000, "r--", linewidth=2, label="Ni I 6768 (g=1.43, MDI)")
ax.axhline(y=hmi_bandpass * 1000, color="green", linestyle=":",
           linewidth=2, label="HMI bandpass (76 mA)")

# Find critical B where splitting = bandpass
B_crit_fe = hmi_bandpass / (4.67e-13 * fe_lambda**2 * fe_g)
B_crit_ni = hmi_bandpass / (4.67e-13 * ni_lambda**2 * ni_g)
ax.axvline(x=B_crit_fe, color="blue", linestyle=":", alpha=0.5)
ax.axvline(x=B_crit_ni, color="red", linestyle=":", alpha=0.5)

ax.set_xlabel("Magnetic Field B [Gauss]")
ax.set_ylabel("Zeeman Splitting [mA]")
ax.set_title("Zeeman Splitting vs Magnetic Field Strength")
ax.legend(fontsize=10)
ax.set_xlim(0, 5000)
plt.tight_layout()
plt.show()

# Print critical values
print("Critical B (splitting = HMI bandpass 76 mA):")
print("  Fe I 6173 (HMI): B = {:.0f} G".format(B_crit_fe))
print("  Ni I 6768 (MDI): B = {:.0f} G".format(B_crit_ni))
print("")
print("At B = 1000 G:")
print("  Fe I splitting: {:.1f} mA".format(zeeman_splitting_nm(fe_lambda, fe_g, 1000) * 1000))
print("  Ni I splitting: {:.1f} mA".format(zeeman_splitting_nm(ni_lambda, ni_g, 1000) * 1000))
print("")
print("Fe I 6173 advantage: {:.1f}x more sensitive".format(
    (fe_lambda**2 * fe_g) / (ni_lambda**2 * ni_g)))

## Part 3: Stokes 프로파일 / Stokes Profiles

Stokes 벡터 (I, Q, U, V)는 편광 상태를 완전히 기술합니다.
The Stokes vector (I, Q, U, V) completely describes the polarization state.

- **I**: 총 강도 / Total intensity
- **Q, U**: 선형 편광 / Linear polarization (transverse B)
- **V**: 원형 편광 / Circular polarization (line-of-sight B)

In [ ]:
# =============================================================
# Stokes Profiles from Zeeman Splitting
# =============================================================

def gaussian(x, center, sigma, depth):
    """Simple Gaussian absorption profile."""
    return depth * np.exp(-0.5 * ((x - center) / sigma)**2)


def stokes_profiles(wavelength, B_gauss, gamma=0.0, chi=0.0,
                    lambda0=617.3, g_eff=2.5, sigma=0.005, depth=0.7):
    """Compute Stokes I, Q, U, V for normal Zeeman triplet.

    Args:
        wavelength: Wavelength array in nm.
        B_gauss: Magnetic field strength in Gauss.
        gamma: Inclination angle (0 = along LOS) in radians.
        chi: Azimuth angle in radians.
        lambda0: Central wavelength in nm.
        g_eff: Effective Lande g-factor.
        sigma: Line width in nm.
        depth: Line depth (0-1).

    Returns:
        Tuple of (I, Q, U, V) arrays.
    """
    dL = zeeman_splitting_nm(lambda0, g_eff, B_gauss)

    # Zeeman components: sigma-, pi, sigma+
    phi_minus = gaussian(wavelength, lambda0 - dL, sigma, depth)
    phi_pi = gaussian(wavelength, lambda0, sigma, depth)
    phi_plus = gaussian(wavelength, lambda0 + dL, sigma, depth)

    # Stokes parameters (simplified radiative transfer)
    sin2g = np.sin(gamma)**2
    cos_g = np.cos(gamma)

    I = 1.0 - 0.5 * (phi_plus + phi_minus) * (1 + cos_g**2) / 2 - phi_pi * sin2g / 2
    Q = (phi_pi - 0.5 * (phi_plus + phi_minus)) * sin2g * np.cos(2 * chi) * 0.5
    U = (phi_pi - 0.5 * (phi_plus + phi_minus)) * sin2g * np.sin(2 * chi) * 0.5
    V = 0.5 * (phi_minus - phi_plus) * cos_g

    return I, Q, U, V


# Wavelength grid centered on Fe I 6173
wav = np.linspace(617.25, 617.35, 500)

# Different magnetic field strengths
B_values = [200, 500, 1000, 2000]
colors = ["#2196F3", "#4CAF50", "#FF9800", "#E91E63"]
gamma = np.pi / 4  # 45 degree inclination
chi = np.pi / 6    # 30 degree azimuth

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
labels = ["Stokes I", "Stokes Q", "Stokes U", "Stokes V"]

for i, B_val in enumerate(B_values):
    I, Q, U, V = stokes_profiles(wav, B_val, gamma=gamma, chi=chi)
    lbl = "B = {} G".format(B_val)
    axes[0, 0].plot(wav, I, color=colors[i], label=lbl)
    axes[0, 1].plot(wav, Q, color=colors[i], label=lbl)
    axes[1, 0].plot(wav, U, color=colors[i], label=lbl)
    axes[1, 1].plot(wav, V, color=colors[i], label=lbl)

for idx, ax in enumerate(axes.flat):
    ax.set_title(labels[idx])
    ax.legend(fontsize=8)
    ax.axhline(y=0 if idx > 0 else 1, color="gray", linestyle="--", alpha=0.3)
    ax.set_ylabel("Normalized Intensity")
for ax in axes[1]:
    ax.set_xlabel("Wavelength [nm]")

fig.suptitle("Stokes Profiles: Normal Zeeman Triplet (Fe I 6173 A)", fontsize=13)
plt.tight_layout()
plt.show()

print("Key features:")
print("  V profile: antisymmetric -- sensitive to LOS magnetic field")
print("  Q/U profiles: symmetric -- sensitive to transverse field")
print("  Larger B -> wider splitting and stronger polarization signals")

## Part 4: HMI 6점 파장 샘플링 / HMI 6-Point Wavelength Sampling

HMI는 Fe I 6173 A 흡수선을 6개 파장 위치에서 샘플링합니다.
HMI samples the Fe I 6173 A absorption line at 6 wavelength positions.

샘플 위치: 중심으로부터 +/-34.5, +/-103.5, +/-172.5 mA
Sample positions: +/-34.5, +/-103.5, +/-172.5 mA from line center

In [ ]:
# =============================================================
# HMI 6-Point Wavelength Sampling
# =============================================================

# Fe I 6173 line parameters
lambda0 = 617.3  # nm
line_sigma = 0.005  # nm (line width)
line_depth = 0.7

# HMI sample positions in mA from line center
sample_offsets_mA = np.array([-172.5, -103.5, -34.5, 34.5, 103.5, 172.5])
sample_offsets_nm = sample_offsets_mA * 1e-4  # convert mA to nm

# Wavelength grid
wav = np.linspace(617.24, 617.36, 1000)

# Line profiles: rest and Doppler-shifted
rest_profile = 1.0 - gaussian(wav, lambda0, line_sigma, line_depth)
v_shift = 2.0  # km/s
doppler_shift = lambda0 * v_shift / 3e5  # nm
shifted_profile = 1.0 - gaussian(wav, lambda0 + doppler_shift, line_sigma, line_depth)

fig, ax = plt.subplots(figsize=(10, 6))

# Plot line profiles
ax.plot(wav, rest_profile, "b-", linewidth=2, label="Rest (v = 0)")
ax.plot(wav, shifted_profile, "r--", linewidth=2, label="Doppler shifted (v = 2 km/s)")

# Mark sample positions
sample_wavs = lambda0 + sample_offsets_nm
sample_intensities = 1.0 - gaussian(sample_wavs, lambda0, line_sigma, line_depth)
shifted_intensities = 1.0 - gaussian(sample_wavs, lambda0 + doppler_shift, line_sigma, line_depth)

for j in range(6):
    color = "blue" if j == 0 else None
    ax.axvline(x=sample_wavs[j], color="gray", linestyle=":", alpha=0.5)
    ax.plot(sample_wavs[j], sample_intensities[j], "bo", markersize=10,
            zorder=5, label="Sample points" if j == 0 else None)
    ax.plot(sample_wavs[j], shifted_intensities[j], "r^", markersize=8, zorder=5)

# Label sample positions
for j, offset in enumerate(sample_offsets_mA):
    y_pos = 0.24 if j % 2 == 0 else 0.22
    ax.text(sample_wavs[j], y_pos, "{:+.1f}".format(offset),
            ha="center", fontsize=8, rotation=45)

ax.set_xlabel("Wavelength [nm]")
ax.set_ylabel("Normalized Intensity")
ax.set_title("HMI 6-Point Wavelength Sampling of Fe I 6173 A")
ax.legend()
plt.tight_layout()
plt.show()

# Simple velocity estimation from intensity ratios
# Using ratio of blue to red wing intensities
I_blue = sample_intensities[0] + sample_intensities[1] + sample_intensities[2]
I_red = sample_intensities[3] + sample_intensities[4] + sample_intensities[5]
I_blue_s = shifted_intensities[0] + shifted_intensities[1] + shifted_intensities[2]
I_red_s = shifted_intensities[3] + shifted_intensities[4] + shifted_intensities[5]

print("Velocity estimation from blue/red wing ratio:")
print("  Rest: blue/red = {:.4f} (symmetric)".format(I_blue / I_red))
print("  Shifted: blue/red = {:.4f} (asymmetric)".format(I_blue_s / I_red_s))
print("  Doppler shift applied: {:.4f} nm = {:.1f} mA".format(
    doppler_shift, doppler_shift * 1e4))

## Part 5: Stokes 복조 / Stokes Demodulation

HMI는 6개 편광 상태를 순환하며 측정한 후, 복조 행렬을 사용하여 Stokes 벡터를 복원합니다.
HMI cycles through 6 polarization states and uses a demodulation matrix to recover the Stokes vector.

변조 행렬의 pseudo-inverse를 통해 Stokes 파라미터를 추출합니다.
Stokes parameters are extracted via the pseudo-inverse of the modulation matrix.

In [ ]:
# =============================================================
# Polarization Modulation and Demodulation
# =============================================================

# HMI-like 6-state modulation matrix (simplified)
# Each row = one polarization state measurement
# Columns = [I, Q, U, V] contributions
# Based on rotating waveplate + polarizer scheme
angles = np.array([0, 60, 120, 30, 90, 150])  # waveplate angles in degrees
theta = np.deg2rad(angles)

# Modulation matrix: M[i,:] gives the sensitivity of measurement i to [I,Q,U,V]
M = np.zeros((6, 4))
for k in range(6):
    M[k, 0] = 1.0                           # I always contributes
    M[k, 1] = np.cos(2 * theta[k])**2       # Q sensitivity
    M[k, 2] = np.cos(2 * theta[k]) * np.sin(2 * theta[k])  # U sensitivity
    M[k, 3] = (-1)**k * 0.5                  # V sensitivity (RCP/LCP switching)

print("Modulation Matrix M (6 measurements x 4 Stokes):")
print("-" * 50)
header = "  {:>8s} {:>8s} {:>8s} {:>8s}"
print(header.format("I", "Q", "U", "V"))
for row in M:
    print("  {:8.3f} {:8.3f} {:8.3f} {:8.3f}".format(*row))

# Demodulation matrix via pseudo-inverse
D = np.linalg.pinv(M)
print("")
print("Demodulation Matrix D = pinv(M) (4 Stokes x 6 measurements):")
print("-" * 65)
for i, name in enumerate(["I", "Q", "U", "V"]):
    vals = "  ".join(["{:7.3f}".format(v) for v in D[i]])
    print("  " + name + ": " + vals)

# Round-trip test: input Stokes -> modulate -> demodulate -> recovered
input_stokes = np.array([1.0, 0.05, -0.03, 0.10])
measurements = M @ input_stokes
noise = np.random.normal(0, 0.001, 6)
noisy_measurements = measurements + noise
recovered = D @ noisy_measurements

print("")
print("Round-trip test:")
print("  Input Stokes:     I={:.4f}  Q={:.4f}  U={:.4f}  V={:.4f}".format(*input_stokes))
print("  Recovered Stokes: I={:.4f}  Q={:.4f}  U={:.4f}  V={:.4f}".format(*recovered))
print("  Error:            I={:.5f}  Q={:.5f}  U={:.5f}  V={:.5f}".format(
    *(recovered - input_stokes)))

# SNR analysis
print("")
print("Polarization sensitivity (weak-field regime):")
print("  V (circular) ~ B * cos(gamma)     -> linear in B_LOS")
print("  Q,U (linear) ~ B^2 * sin^2(gamma) -> quadratic in B_trans")
print("  => V signal dominates for weak fields (B < 500 G)")
print("  => Q,U require stronger fields or longer integration")

## Part 6: Milne-Eddington 역산 개념 / ME Inversion Concept

HMI VFISV 코드는 Milne-Eddington 대기 모델을 사용하여 Stokes 프로파일을 피팅합니다.
The HMI VFISV code uses a Milne-Eddington atmospheric model to fit observed Stokes profiles.

ME 모델은 source function이 optical depth에 대해 선형이라고 가정합니다.
The ME model assumes the source function is linear with optical depth.

In [ ]:
# =============================================================
# Simplified Milne-Eddington Forward Model
# =============================================================

def voigt_profile(x, sigma, gamma_v):
    """Compute Voigt profile using scipy if available.

    Args:
        x: Frequency/wavelength offset array.
        sigma: Gaussian width.
        gamma_v: Lorentzian width.

    Returns:
        Voigt profile values.
    """
    try:
        return special.voigt_profile(x, sigma, gamma_v)
    except AttributeError:
        # Fallback to Gaussian approximation
        return np.exp(-0.5 * (x / sigma)**2) / (sigma * np.sqrt(2 * np.pi))


def me_stokes(wav, B, gamma, chi, eta0=8.0, dlamD=0.003, a=0.05,
              S0=0.2, S1=0.8, lambda0=617.3, g_eff=2.5):
    """Simplified Milne-Eddington Stokes profiles.

    Args:
        wav: Wavelength array in nm.
        B: Magnetic field in Gauss.
        gamma: Inclination angle in radians.
        chi: Azimuth angle in radians.
        eta0: Line-to-continuum opacity ratio.
        dlamD: Doppler width in nm.
        a: Damping parameter.
        S0: Source function constant term.
        S1: Source function gradient.
        lambda0: Central wavelength in nm.
        g_eff: Effective Lande g-factor.

    Returns:
        Tuple of (I, Q, U, V) arrays.
    """
    dL = zeeman_splitting_nm(lambda0, g_eff, B)
    v = (wav - lambda0) / dlamD

    # Voigt profiles for the three Zeeman components
    v_shift = dL / dlamD
    H_pi = voigt_profile(v * dlamD, dlamD / np.sqrt(2), a * dlamD)
    H_plus = voigt_profile((v - v_shift) * dlamD, dlamD / np.sqrt(2), a * dlamD)
    H_minus = voigt_profile((v + v_shift) * dlamD, dlamD / np.sqrt(2), a * dlamD)

    # Normalize
    H_pi = H_pi / H_pi.max()
    H_plus = H_plus / H_plus.max() if H_plus.max() > 0 else H_plus
    H_minus = H_minus / H_minus.max() if H_minus.max() > 0 else H_minus

    # Absorption profiles
    sin2g = np.sin(gamma)**2
    cos_g = np.cos(gamma)

    eta_I = eta0 * (H_pi * sin2g / 2 + (H_plus + H_minus) * (1 + cos_g**2) / 4)
    eta_Q = eta0 * (H_pi - (H_plus + H_minus) / 2) * sin2g * np.cos(2 * chi) / 2
    eta_U = eta0 * (H_pi - (H_plus + H_minus) / 2) * sin2g * np.sin(2 * chi) / 2
    eta_V = eta0 * (H_minus - H_plus) * cos_g / 2

    # Milne-Eddington solution
    Delta = (1 + eta_I)**2 * ((1 + eta_I)**2 - eta_Q**2 - eta_U**2 - eta_V**2)
    Delta = np.maximum(Delta, 1e-10)

    I_out = S0 + S1 * (1 + eta_I) * ((1 + eta_I)**2 - eta_Q**2 - eta_U**2 - eta_V**2) / Delta
    Q_out = -S1 * eta_Q * (1 + eta_I)**2 / Delta
    U_out = -S1 * eta_U * (1 + eta_I)**2 / Delta
    V_out = -S1 * eta_V * (1 + eta_I)**2 / Delta

    return I_out, Q_out, U_out, V_out


# Generate profiles for three field strengths
wav = np.linspace(617.24, 617.36, 500)
cases = [
    (100, "Weak: B=100 G"),
    (1000, "Moderate: B=1000 G"),
    (3000, "Strong: B=3000 G"),
]
gamma_val = np.pi / 4
chi_val = np.pi / 6
colors = ["#2196F3", "#FF9800", "#E91E63"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
titles = ["Stokes I (ME)", "Stokes Q (ME)", "Stokes U (ME)", "Stokes V (ME)"]

for idx, (B_val, label) in enumerate(cases):
    I_me, Q_me, U_me, V_me = me_stokes(wav, B_val, gamma_val, chi_val)
    axes[0, 0].plot(wav, I_me, color=colors[idx], label=label, linewidth=1.5)
    axes[0, 1].plot(wav, Q_me, color=colors[idx], label=label, linewidth=1.5)
    axes[1, 0].plot(wav, U_me, color=colors[idx], label=label, linewidth=1.5)
    axes[1, 1].plot(wav, V_me, color=colors[idx], label=label, linewidth=1.5)

for i, ax in enumerate(axes.flat):
    ax.set_title(titles[i])
    ax.legend(fontsize=8)
    ax.set_ylabel("Intensity")
for ax in axes[1]:
    ax.set_xlabel("Wavelength [nm]")

fig.suptitle("Milne-Eddington Model: Synthetic Stokes Profiles", fontsize=13)
plt.tight_layout()
plt.show()

## Part 7: 180도 비명확성 / 180-Degree Disambiguation

Stokes Q와 U는 $\cos(2\chi)$, $\sin(2\chi)$에 의존하므로, $\chi$와 $\chi + 180°$를 구별할 수 없습니다.
Since Stokes Q and U depend on $\cos(2\chi)$ and $\sin(2\chi)$, angles $\chi$ and $\chi + 180°$ are indistinguishable.

이를 해결하기 위해 자기장의 발산을 최소화하는 방향을 선택합니다 ($\nabla \cdot \mathbf{B} = 0$).
To resolve this, we choose the direction that minimizes the divergence of B ($\nabla \cdot \mathbf{B} = 0$).

In [ ]:
# =============================================================
# 180-Degree Disambiguation
# =============================================================

# Part A: Show that cos(2*chi) = cos(2*(chi+pi))
chi_test = np.linspace(0, 2 * np.pi, 360)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Polar plot showing ambiguity
ax1 = plt.subplot(121, projection="polar")
r1 = np.ones(100)
theta1 = np.linspace(0, 2 * np.pi, 100)
ax1.plot(theta1, r1, "b-", alpha=0.3)

# Two ambiguous directions
phi_true = np.deg2rad(45)
phi_ambig = phi_true + np.pi
ax1.annotate("", xy=(phi_true, 1.0), xytext=(0, 0),
             arrowprops=dict(arrowstyle="->", color="blue", lw=2))
ax1.annotate("", xy=(phi_ambig, 1.0), xytext=(0, 0),
             arrowprops=dict(arrowstyle="->", color="red", lw=2))
ax1.set_title("Two Possible B_trans Directions", pad=20)

# Part B: Simple divergence minimization on a 2D grid
ax2 = axes[1]
np.random.seed(42)
nx, ny = 6, 6
# True field: smooth radial pattern
x_grid, y_grid = np.meshgrid(np.arange(nx), np.arange(ny))
Bx_true = x_grid - nx / 2
By_true = y_grid - ny / 2

# Ambiguous: randomly flip 180 degrees in some pixels
flip_mask = np.random.choice([1, -1], size=(ny, nx))
Bx_amb = Bx_true * flip_mask
By_amb = By_true * flip_mask

# Simple greedy disambiguation: minimize local divergence
Bx_fix = Bx_amb.copy()
By_fix = By_amb.copy()
for iteration in range(10):
    for i in range(1, ny - 1):
        for j in range(1, nx - 1):
            # Compute divergence with current direction
            div_current = abs(Bx_fix[i, j+1] - Bx_fix[i, j-1]) + abs(By_fix[i+1, j] - By_fix[i-1, j])
            # Compute divergence with flipped direction
            div_flipped = abs(-Bx_fix[i, j] - Bx_fix[i, j-1] + Bx_fix[i, j+1]) + \
                          abs(-By_fix[i, j] - By_fix[i-1, j] + By_fix[i+1, j])
            if div_flipped < div_current:
                Bx_fix[i, j] *= -1
                By_fix[i, j] *= -1

ax2.quiver(x_grid, y_grid, Bx_amb, By_amb, color="red", alpha=0.4, label="Ambiguous")
ax2.quiver(x_grid, y_grid, Bx_fix, By_fix, color="blue", alpha=0.8, label="Disambiguated")
ax2.set_title("Divergence Minimization")
ax2.legend(fontsize=9)
ax2.set_aspect("equal")

plt.tight_layout()
plt.show()

# Mathematical proof
print("Mathematical proof of 180-degree ambiguity:")
print("  cos(2*chi) = cos(2*(chi + pi))")
print("  cos(2*chi) = cos(2*chi + 2*pi) = cos(2*chi)  [QED]")
print("")
print("  sin(2*chi) = sin(2*(chi + pi))")
print("  sin(2*chi) = sin(2*chi + 2*pi) = sin(2*chi)  [QED]")
print("")
print("Therefore Q and U cannot distinguish chi from chi + 180 deg.")

## Part 8: 데이터 제품 / Data Products

HMI는 초당 대량의 데이터를 생성하며, 다양한 수준의 데이터 제품을 제공합니다.
HMI generates massive amounts of data per second and provides various levels of data products.

- **Level 0**: Raw filtergrams / 원시 필터그램
- **Level 1**: Calibrated images / 보정된 이미지
- **Level 1.5**: Observables (Dopplergrams, magnetograms) / 관측량
- **Level 2**: Inverted vector fields / 역산된 벡터장

In [ ]:
# =============================================================
# HMI Data Products and Volume Calculations
# =============================================================

# Image parameters
npix = 4096
bytes_per_pixel = 2  # 16-bit CCD
image_size_bytes = npix * npix * bytes_per_pixel
image_size_MB = image_size_bytes / 1e6

# Filtergram cadence
filtergrams_per_set = 36  # 6 wavelengths x 6 polarization states
set_cadence_s = 135  # seconds for one complete set (vector mode)
images_per_day = filtergrams_per_set * (86400 / set_cadence_s)
daily_volume_GB = images_per_day * image_size_MB / 1000

print("=" * 55)
print("HMI Data Volume Calculations")
print("=" * 55)
print("  Image size: {} x {} x {} bytes = {:.1f} MB".format(
    npix, npix, bytes_per_pixel, image_size_MB))
print("  Filtergrams per set: {}".format(filtergrams_per_set))
print("  Set cadence: {} s".format(set_cadence_s))
print("  Images per day: {:.0f}".format(images_per_day))
print("  Daily raw volume: {:.0f} GB".format(daily_volume_GB))
print("  Telemetry rate: 55 Mbps")
print("=" * 55)

# Data product summary
products = [
    ("Dopplergram", "45 s", "LOS velocity"),
    ("LOS Magnetogram", "45 s", "B_LOS (Mx/cm2)"),
    ("Continuum Intensity", "45 s", "Ic"),
    ("Line-depth Image", "45 s", "Id"),
    ("Line-width Image", "45 s", "Iw"),
    ("Vector Magnetogram", "720 s", "B, gamma, chi"),
]
print("")
print("HMI Data Products:")
print("-" * 55)
print("{:<22s} {:<10s} {:<20s}".format("Product", "Cadence", "Content"))
print("-" * 55)
for name, cadence, content in products:
    print("{:<22s} {:<10s} {:<20s}".format(name, cadence, content))
print("-" * 55)

# Cadence comparison bar chart
fig, ax = plt.subplots(figsize=(9, 5))
product_names = [p[0] for p in products]
cadences = [45, 45, 45, 45, 45, 720]
colors = ["#2196F3"] * 5 + ["#E91E63"]
bars = ax.barh(range(len(products)), cadences, color=colors, edgecolor="black")
ax.set_yticks(range(len(products)))
ax.set_yticklabels(product_names)
ax.set_xlabel("Cadence [seconds]")
ax.set_title("HMI Data Product Cadences")

for bar, val in zip(bars, cadences):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height() / 2,
            str(val) + " s", va="center")

plt.tight_layout()
plt.show()

## Part 9: 일진동학 개념 / Helioseismology Concepts

HMI의 4096x4096 해상도는 MDI(1024x1024)보다 높은 구면 조화 차수(l)까지 분해할 수 있습니다.
HMI's 4096x4096 resolution can resolve higher spherical harmonic degrees (l) than MDI (1024x1024).

$$l_{\max} \approx \frac{\pi R_\odot}{\text{pixel size}}$$

In [ ]:
# =============================================================
# Helioseismology: Synthetic l-nu Diagram
# =============================================================

# Solar p-mode frequencies (simplified model)
# nu_{n,l} ~ Delta_nu * (n + l/2 + epsilon) - D0 * l*(l+1)
Delta_nu = 135.0  # microHz, large frequency separation
epsilon = 1.5     # phase offset
D0 = 1.5          # small separation coefficient

# Generate mode frequencies
l_modes = np.arange(0, 300)
n_modes = np.arange(1, 30)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: l-nu diagram
ax1 = axes[0]
for n in n_modes:
    nu = Delta_nu * (n + l_modes / 2 + epsilon) - D0 * l_modes * (l_modes + 1) / (n + l_modes / 2)
    # Only plot physically reasonable frequencies
    valid = (nu > 1000) & (nu < 5500)
    # Add scatter to simulate real power
    power = np.exp(-((nu - 3100) / 800)**2)  # p-mode envelope
    sizes = power[valid] * 3 + 0.3
    ax1.scatter(l_modes[valid], nu[valid] / 1000, s=sizes, c="navy", alpha=0.3)

ax1.set_xlabel("Spherical Harmonic Degree l")
ax1.set_ylabel("Frequency [mHz]")
ax1.set_title("Synthetic l-nu Diagram (p-modes)")
ax1.set_xlim(0, 250)
ax1.set_ylim(1, 5.5)

# Mark MDI and HMI l_max
R_sun_arcsec = 960  # arcsec
mdi_pixel = 2.0  # arcsec/pixel (full-disk)
hmi_pixel = 0.5  # arcsec/pixel

l_max_mdi = int(np.pi * R_sun_arcsec / mdi_pixel)
l_max_hmi = int(np.pi * R_sun_arcsec / hmi_pixel)

ax1.axvline(x=l_max_mdi, color="red", linestyle="--", linewidth=2,
            label="MDI l_max ~ {}".format(l_max_mdi))
ax1.axvline(x=200, color="blue", linestyle="--", linewidth=2,
            label="HMI practical ~ 200")
ax1.legend(fontsize=9)

# Right: Resolution comparison
ax2 = axes[1]
instruments = ["MDI\n(1024x1024)", "HMI\n(4096x4096)"]
l_max_values = [l_max_mdi, min(l_max_hmi, 4000)]
pixel_sizes = [mdi_pixel, hmi_pixel]

bars = ax2.bar(instruments, l_max_values, color=["#FF9800", "#2196F3"],
               edgecolor="black", width=0.5)
for bar, val in zip(bars, l_max_values):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
             "l_max = {}".format(val), ha="center", fontweight="bold")

ax2.set_ylabel("Maximum l (theoretical)")
ax2.set_title("Spatial Resolution: l_max Comparison")
ax2.set_yscale("log")

plt.tight_layout()
plt.show()

print("Resolution comparison:")
print("  MDI: {} arcsec/pixel -> l_max ~ {}".format(mdi_pixel, l_max_mdi))
print("  HMI: {} arcsec/pixel -> l_max ~ {}".format(hmi_pixel, l_max_hmi))
print("  Practical HMI limit: l ~ 200 (noise considerations)")
print("  Improvement factor: {:.0f}x in l_max".format(l_max_hmi / l_max_mdi))

## Part 10: 기기 발전사 / Instrument Evolution Timeline

태양 자기장 관측 기기의 발전 역사와 데이터 전송률의 진화를 보여줍니다.
Shows the evolution of solar magnetograph instruments and the growth in data telemetry rates.

In [ ]:
# =============================================================
# Solar Magnetograph Instrument Timeline
# =============================================================

instruments = [
    ("Mt. Wilson", 1962, 0.001, "Ground"),
    ("Kitt Peak", 1974, 0.01, "Ground"),
    ("GONG", 1995, 0.1, "Ground network"),
    ("SOHO/MDI", 1995, 5.0, "L1 orbit"),
    ("Hinode/SOT", 2006, 2.0, "Sun-sync orbit"),
    ("SDO/HMI", 2010, 55.0, "GEO orbit"),
]

years = [inst[1] for inst in instruments]
names = [inst[0] for inst in instruments]
rates = [inst[2] for inst in instruments]
locations = [inst[3] for inst in instruments]

fig, axes = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={"height_ratios": [1, 1.2]})

# Top panel: Timeline
ax1 = axes[0]
ax1.set_xlim(1955, 2015)
ax1.set_ylim(-1, 1)
ax1.axhline(y=0, color="black", linewidth=2)

for i, (name, year, rate, loc) in enumerate(instruments):
    color = "#E91E63" if "SDO" in name else "#2196F3"
    marker_size = 15 if "SDO" in name else 10
    ax1.plot(year, 0, "o", color=color, markersize=marker_size, zorder=5)

    y_offset = 0.4 if i % 2 == 0 else -0.5
    ax1.annotate(
        name + " (" + str(year) + ")",
        xy=(year, 0),
        xytext=(year, y_offset),
        fontsize=9,
        fontweight="bold" if "SDO" in name else "normal",
        ha="center",
        arrowprops=dict(arrowstyle="-", color="gray", alpha=0.5),
    )

ax1.set_yticks([])
ax1.set_xlabel("Year")
ax1.set_title("Solar Magnetograph Instruments Timeline")

# Bottom panel: Data rate evolution (log scale)
ax2 = axes[1]
ax2.bar(range(len(instruments)), rates, color=["#90CAF9"] * 5 + ["#E91E63"],
        edgecolor="black", width=0.6)
ax2.set_xticks(range(len(instruments)))
labels = ["{}\n({})".format(n, y) for n, y in zip(names, years)]
ax2.set_xticklabels(labels, fontsize=8)
ax2.set_ylabel("Telemetry Rate [Mbps]")
ax2.set_title("Data Rate Evolution (Log Scale)")
ax2.set_yscale("log")

for i, rate in enumerate(rates):
    ax2.text(i, rate * 1.3, "{} Mbps".format(rate), ha="center", fontsize=8)

plt.tight_layout()
plt.show()

print("Key milestones:")
print("  1962: Mt. Wilson - first routine magnetograms")
print("  1974: Kitt Peak - systematic full-disk observations")
print("  1995: GONG + MDI - continuous helioseismology begins")
print("  2006: Hinode/SOT - high-res spectropolarimetry (0.3 arcsec)")
print("  2010: SDO/HMI - full-disk vector magnetometry at 1 arcsec")
print("")
print("Data rate growth: {:.0f}x from MDI to HMI".format(55.0 / 5.0))